# 00 · Common library notebook

**Project:** Inventory Demand Forecasting via Kalman Filtering and Global DeepAR Models — *Phase 2 (Deep Learning)*

This notebook is **not run on its own**. Every experiment notebook loads it with
`%run ./00_common.ipynb`, so all experiments use *exactly the same* data windows, model code and metrics.
That is what makes the comparison fair.

| Section | What it contains | Owner |
|---|---|---|
| A | Paths, configuration, seeds | Harsha |
| B | Loading the processed panel + building DL training windows | Saumya |
| C | Kalman filter (local level model) | Harshvardhan |
| D | DeepAR model, likelihoods, training loop, sampling | Shivansh |
| E | Kalman → DeepAR hybrid features | Krishiv |
| F | Evaluation metrics (MAE, RMSE, WAPE, pinball, coverage) | Harsha |

## Section A — Configuration

In [ ]:
import os, math, time, json, random, warnings
from pathlib import Path

# macOS fix: PyTorch and LightGBM each ship their own OpenMP runtime (libomp). When both are loaded in one
# process the kernel aborts with "OMP: Error #15" ("Kernel died"). Allowing the duplicate runtime avoids it.
# It must be set before torch / lightgbm are imported. Harmless on Linux, Windows and Colab.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
torch.set_num_threads(max(1, os.cpu_count() or 1))

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
FC, TAB, FIG = ROOT / "results" / "forecasts", ROOT / "results" / "tables", ROOT / "figures"
for _p in (PROC, FC, TAB, FIG):
    _p.mkdir(parents=True, exist_ok=True)

CFG = dict(
    store="CA_1",
    depts=["FOODS_3", "HOUSEHOLD_1", "HOBBIES_1"],
    items_per_dept=100,          # 300 SKUs in total
    start_d=942, end_d=1941,     # last 1000 days of M5 (2013-08 .. 2016-05)
    horizon=28,                  # forecast 28 days ahead (M5 standard)
    context=56,                  # DeepAR conditioning range (8 weeks)
    origins={"O1": 1857, "O2": 1885, "O3": 1913},  # last observed M5 day of each rolling origin
    quantiles=[0.1, 0.5, 0.9],   # 80% prediction interval + median
    seed=42,
)
H, C, QUANTILES = CFG["horizon"], CFG["context"], CFG["quantiles"]
ORIGINS = {k: v - CFG["start_d"] for k, v in CFG["origins"].items()}   # as time index t (0..999)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


# Default hyper-parameters of the global model (Section D). Ablations override single keys.
MODEL_CFG = dict(
    cell="lstm", likelihood="nb", hidden=40, layers=2, emb_item=10, emb_dept=2,
    dropout=0.1, weight_decay=1e-4, item_drop=0.1,       # regularisation
    use_item=True, use_cov=True, use_kf=False, mask_stockouts=True,
    lr=2e-3, epochs=40, patience=5, windows_per_series=16, batch_size=128, clip=10.0,
)

## Section B — Panel loading and DL windows 

The processed panel (built in `01_data_pipeline.ipynb`) is stored as dense arrays of shape
`N series × T days`:

* `Y` – unit sales; **NaN** when the item is not on sale (not launched / store closed),
* `LISTED` – 1 if the item has a price that week (known in advance),
* `STOCK` – 1 if the day is a *probable stockout* (detected causally, never imputed),
* `X` – known-in-advance covariates (calendar, SNAP, events, price, age).

A **training window** is a slice of length `C + H` around a split point `t0`:
positions `t0-C+1 … t0` form the *context*, `t0+1 … t0+H` the *prediction range*.
The network input at time `t` uses `y[t-1]` (the previous value), never `y[t]` — this is what
prevents look-ahead.

In [ ]:
def load_panel():
    z = np.load(PROC / "panel.npz", allow_pickle=True)
    P = {k: z[k] for k in z.files}
    P["meta"] = pd.read_csv(PROC / "series_meta.csv")
    P["N"], P["T"] = P["Y"].shape
    return P


def window_index(t0, length_before=C, length_after=H):
    """Time indices of a window: shape (B, C+H)."""
    t0 = np.asarray(t0)
    return t0[:, None] + np.arange(-length_before + 1, length_after + 1)[None, :]


def make_batch(P, sidx, t0, mcfg, kf=None, item_override=None):
    """Build one batch of DeepAR windows as torch tensors.

    sidx : (B,) series indices,  t0 : (B,) split points (last observed day).
    Returns a dict; everything is already divided by the per-window scale nu.
    """
    sidx, t0 = np.asarray(sidx), np.asarray(t0)
    tt = window_index(t0)                                  # (B, L)
    s2 = sidx[:, None]
    y = P["Y"][s2, tt]                                     # target (NaN = not on sale)
    y_prev = P["Y"][s2, tt - 1]                           # lagged target = network input
    stock, stock_prev = P["STOCK"][s2, tt], P["STOCK"][s2, tt - 1]

    # scale nu = 1 + mean of observed context values (DeepAR, Salinas et al. 2020)
    ctx = y[:, :C]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        nu = 1.0 + np.nan_to_num(np.nanmean(ctx, axis=1), nan=0.0)

    observed = ~np.isnan(y)
    loss_mask = observed & (stock == 0) if mcfg["mask_stockouts"] else observed
    prev_obs = (~np.isnan(y_prev)).astype(np.float32)

    feats = [np.nan_to_num(y_prev) / nu[:, None], prev_obs, stock_prev.astype(np.float32)]
    dyn = np.stack(feats, axis=-1)                         # (B, L, 3)
    if mcfg["use_cov"]:
        dyn = np.concatenate([dyn, P["X"][s2, tt]], axis=-1)
    if mcfg["use_kf"]:
        dyn = np.concatenate([dyn, kalman_window_features(kf, sidx, t0, tt, nu)], axis=-1)

    item = P["meta"]["item_idx"].values[sidx] if item_override is None else item_override
    return dict(
        dyn=torch.tensor(dyn, dtype=torch.float32),
        y=torch.tensor(np.nan_to_num(y), dtype=torch.float32),
        mask=torch.tensor(loss_mask, dtype=torch.float32),
        nu=torch.tensor(nu, dtype=torch.float32),
        item=torch.tensor(item, dtype=torch.long),
        dept=torch.tensor(P["meta"]["dept_idx"].values[sidx], dtype=torch.long),
        listed=torch.tensor(P["LISTED"][s2, tt], dtype=torch.float32),
        sidx=sidx, t0=t0,
    )


def sample_train_windows(P, series, last_t, k, rng):
    """k random split points per series such that the whole window ends on/before last_t."""
    lo, hi = C, last_t - H                                  # need t0-C >= 0 and t0+H <= last_t
    sidx = np.repeat(series, k)
    t0 = rng.integers(lo, hi + 1, size=len(sidx))
    return sidx, t0

## Section C — Kalman filter: local level model 

Per SKU we de-seasonalise with a day-of-week profile, $z_t = y_t / s_{dow(t)}$, and track the level with

$$x_t = x_{t-1} + w_t,\; w_t\sim\mathcal N(0,q) \qquad z_t = x_t + v_t,\; v_t \sim \mathcal N(0,r)$$

Predict: $\hat x_{t|t-1}=\hat x_{t-1|t-1}$, $P_{t|t-1}=P_{t-1|t-1}+q$.
Update: $K_t = P_{t|t-1}/(P_{t|t-1}+r)$, $\hat x_{t|t}=\hat x_{t|t-1}+K_t(z_t-\hat x_{t|t-1})$, $P_{t|t}=(1-K_t)P_{t|t-1}$.

Missing days (stockouts, store closed, not launched) skip the update — **the filter marks rather than imputes**.
We write $q=\rho r$ and estimate the signal-to-noise ratio $\rho$ by maximising the Gaussian likelihood
(prediction-error decomposition) with $r$ concentrated out.

In [ ]:
RHO_GRID = np.logspace(-4, 0, 17)


def kalman_local_level(z, rho):
    """Run the filter with r = 1 and q = rho. z may contain NaN (missing).
    Returns filtered level x, variance P (in units of r), innovations v and their variances Fv."""
    T = len(z)
    x, Pv = np.full(T, np.nan), np.full(T, np.nan)
    v, Fv = np.full(T, np.nan), np.full(T, np.nan)
    x_prev = P_prev = None
    for t in range(T):
        if x_prev is None:                    # initialise at the first observation
            if not np.isnan(z[t]):
                x_prev, P_prev = z[t], 1.0
                x[t], Pv[t] = x_prev, P_prev
            continue
        x_pred, P_pred = x_prev, P_prev + rho            # predict
        if np.isnan(z[t]):                                # missing -> no update
            x_prev, P_prev = x_pred, P_pred
        else:
            v[t] = z[t] - x_pred
            Fv[t] = P_pred + 1.0
            K = P_pred / Fv[t]                            # Kalman gain
            x_prev, P_prev = x_pred + K * v[t], (1 - K) * P_pred
        x[t], Pv[t] = x_prev, P_prev
    return x, Pv, v, Fv


def concentrated_loglik(v, Fv):
    ok = ~np.isnan(v)
    if ok.sum() < 5:
        return -np.inf, np.nan
    sigma2 = np.mean(v[ok] ** 2 / Fv[ok])
    ll = -0.5 * (ok.sum() * (np.log(2 * np.pi * sigma2) + 1) + np.log(Fv[ok]).sum())
    return ll, sigma2


def dow_profile(y, dow, kappa=10.0):
    """Multiplicative day-of-week profile, shrunk towards 1 for sparse items (kappa pseudo-days)."""
    m = np.nanmean(y) if np.any(~np.isnan(y)) else 0.0
    if not m > 0:
        return np.ones(7)
    prof = np.ones(7)
    for d in range(7):
        yd = y[(dow == d) & ~np.isnan(y)]
        prof[d] = (yd.sum() + kappa * m) / ((len(yd) + kappa) * m)
    return prof / prof.mean()


def fit_kalman_series(y, dow, fit_end, rho_grid=RHO_GRID):
    """Estimate profile, rho and sigma2 on data up to fit_end (inclusive)."""
    y_fit = y[: fit_end + 1]
    prof = dow_profile(y_fit, dow[: fit_end + 1])
    z = y_fit / prof[dow[: fit_end + 1]]
    best = (-np.inf, rho_grid[0], 1.0)
    for rho in rho_grid:
        _, _, v, Fv = kalman_local_level(z, rho)
        ll, s2 = concentrated_loglik(v, Fv)
        if ll > best[0]:
            best = (ll, rho, s2)
    if not np.isfinite(best[0]):                          # too little data: fall back
        best = (np.nan, 0.05, max(np.nanvar(y_fit), 1.0) if np.any(~np.isnan(y_fit)) else 1.0)
    return dict(prof=prof, rho=best[1], sigma2=best[2], loglik=best[0])


def kalman_forecast(level, Pv, prof, dow_future, q, r, h):
    """h-step-ahead Gaussian forecast from filtered state (level, Pv) at the origin."""
    mean = prof[dow_future] * level
    sd = prof[dow_future] * np.sqrt(Pv + h * q + r)
    return mean, sd

## Section D — DeepAR global model 

One LSTM is shared by **all** SKUs. At each step it receives
`[y_{t-1}/ν, observed_{t-1}, stockout_{t-1}, covariates_t, item embedding, dept embedding]`
and outputs the parameters of a **Negative Binomial** distribution for $y_t$:

$$\mu_t = \nu\,\mathrm{softplus}(w_\mu^\top h_t), \qquad \alpha_t = \mathrm{softplus}(w_\alpha^\top h_t)/\sqrt{\nu}$$

$$\log p(y\mid\mu,\alpha)=\log\Gamma(y+\tfrac1\alpha)-\log\Gamma(y+1)-\log\Gamma(\tfrac1\alpha)
-\tfrac1\alpha\log(1+\alpha\mu)+y\log\tfrac{\alpha\mu}{1+\alpha\mu}$$

Training maximises this log-likelihood with teacher forcing; forecasting draws sample paths (ancestral sampling).

In [ ]:
def nb_log_prob(y, mu, alpha):
    r = 1.0 / alpha
    return (torch.lgamma(y + r) - torch.lgamma(y + 1) - torch.lgamma(r)
            - r * torch.log1p(alpha * mu) + y * torch.log(alpha * mu / (1 + alpha * mu)))


def gauss_log_prob(y, mu, sigma):
    return -0.5 * math.log(2 * math.pi) - torch.log(sigma) - 0.5 * ((y - mu) / sigma) ** 2


class DeepAR(nn.Module):
    def __init__(self, n_items, n_depts, n_dyn, mcfg):
        super().__init__()
        self.mcfg = mcfg
        self.item_emb = nn.Embedding(n_items + 1, mcfg["emb_item"])   # last row = UNKNOWN item (cold start)
        self.dept_emb = nn.Embedding(n_depts, mcfg["emb_dept"])
        self.unk = n_items
        in_dim = n_dyn + mcfg["emb_dept"] + (mcfg["emb_item"] if mcfg["use_item"] else 0)
        rnn_cls = nn.LSTM if mcfg["cell"] == "lstm" else nn.GRU
        self.rnn = rnn_cls(in_dim, mcfg["hidden"], mcfg["layers"], batch_first=True,
                           dropout=mcfg["dropout"] if mcfg["layers"] > 1 else 0.0)
        self.drop = nn.Dropout(mcfg["dropout"])
        self.head = nn.Linear(mcfg["hidden"], 2)

    def static_features(self, item, dept, L):
        parts = [self.dept_emb(dept)]
        if self.mcfg["use_item"]:
            if self.training and self.mcfg["item_drop"] > 0:          # item-ID dropout
                drop = torch.rand(item.shape) < self.mcfg["item_drop"]
                item = torch.where(drop, torch.full_like(item, self.unk), item)
            parts.insert(0, self.item_emb(item))
        return torch.cat(parts, -1).unsqueeze(1).expand(-1, L, -1)

    def forward(self, dyn, item, dept, state=None):
        inp = torch.cat([dyn, self.static_features(item, dept, dyn.shape[1])], -1)
        out, state = self.rnn(inp, state)
        return self.head(self.drop(out)), state

    def dist_params(self, raw, nu):
        nu = nu.view(-1, *([1] * (raw.dim() - 2)))
        if self.mcfg["likelihood"] == "nb":
            mu = F.softplus(raw[..., 0]) * nu + 1e-6
            alpha = F.softplus(raw[..., 1]) / torch.sqrt(nu) + 1e-6
            return mu, alpha
        mu = raw[..., 0] * nu
        sigma = F.softplus(raw[..., 1]) * nu + 1e-6
        return mu, sigma

    def log_prob(self, y, raw, nu):
        a, b = self.dist_params(raw, nu)
        return nb_log_prob(y, a, b) if self.mcfg["likelihood"] == "nb" else gauss_log_prob(y, a, b)

    def sample(self, raw, nu):
        a, b = self.dist_params(raw, nu)
        if self.mcfg["likelihood"] == "nb":
            dist = torch.distributions.NegativeBinomial(total_count=1.0 / b, probs=(a * b / (1 + a * b)).clamp(1e-6, 1 - 1e-6))
            return dist.sample()
        return torch.clamp(torch.normal(a, b), min=0.0)


def masked_nll(model, batch, only_future=False):
    raw, _ = model(batch["dyn"], batch["item"], batch["dept"])
    lp = model.log_prob(batch["y"], raw, batch["nu"])
    mask = batch["mask"].clone()
    if only_future:
        mask[:, :C] = 0
    return -(lp * mask).sum() / mask.sum().clamp(min=1.0)


def train_deepar(P, origin_t, mcfg=None, seed=0, kf=None, exclude=None, verbose=True):
    """Train one global model for a forecast origin.
    Windows end on/before origin-H; the last H days before the origin are the validation range
    used for early stopping, so the test period is never touched."""
    mcfg = {**MODEL_CFG, **(mcfg or {})}
    set_seed(seed)
    rng = np.random.default_rng(seed)
    series = np.setdiff1d(np.arange(P["N"]), np.asarray(exclude if exclude is not None else [], dtype=int))
    n_dyn = make_batch(P, series[:2], np.array([origin_t, origin_t]), mcfg, kf)["dyn"].shape[-1]
    model = DeepAR(P["meta"]["item_idx"].max() + 1, P["meta"]["dept_idx"].max() + 1, n_dyn, mcfg)
    opt = torch.optim.AdamW(model.parameters(), lr=mcfg["lr"], weight_decay=mcfg["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)

    val_t0 = np.full(len(series), origin_t - H)                   # validation: (origin-H, origin]
    val_batch = make_batch(P, series, val_t0, mcfg, kf)
    best, best_state, wait, hist = np.inf, None, 0, []
    for epoch in range(mcfg["epochs"]):
        model.train()
        sidx, t0 = sample_train_windows(P, series, origin_t - H, mcfg["windows_per_series"], rng)
        perm = rng.permutation(len(sidx))
        tr_losses = []
        for i in range(0, len(perm), mcfg["batch_size"]):
            b = perm[i:i + mcfg["batch_size"]]
            batch = make_batch(P, sidx[b], t0[b], mcfg, kf)
            loss = masked_nll(model, batch)
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), mcfg["clip"])   # exploding-gradient guard
            opt.step(); tr_losses.append(loss.item())
        model.eval()
        with torch.no_grad():
            val = masked_nll(model, val_batch, only_future=True).item()
        sched.step(val)
        hist.append(dict(epoch=epoch + 1, train_nll=np.mean(tr_losses), val_nll=val, lr=opt.param_groups[0]["lr"]))
        if val < best - 1e-4:
            best, wait = val, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
        if verbose:
            print(f"epoch {epoch+1:2d}  train NLL {hist[-1]['train_nll']:.4f}  val NLL {val:.4f}  lr {hist[-1]['lr']:.1e}")
        if wait >= mcfg["patience"]:                                      # early stopping
            break
    model.load_state_dict(best_state)
    model.eval()
    return model, pd.DataFrame(hist)


@torch.no_grad()
def forecast_deepar(model, P, origin_t, series=None, n_samples=200, kf=None, item_override=None, seed=0):
    """Sample paths for days origin+1 .. origin+H. Returns array (n_series, n_samples, H)."""
    torch.manual_seed(seed)
    mcfg = model.mcfg
    series = np.arange(P["N"]) if series is None else np.asarray(series)
    batch = make_batch(P, series, np.full(len(series), origin_t), mcfg, kf, item_override)
    dyn, nu, item, dept = batch["dyn"], batch["nu"], batch["item"], batch["dept"]
    _, state = model(dyn[:, :C], item, dept)                  # 1) encode context with true values
    S = n_samples
    rep = lambda x: x.repeat_interleave(S, dim=0)
    if isinstance(state, tuple):                              # LSTM state = (h, c); GRU state = h
        state = tuple(s.repeat_interleave(S, dim=1) for s in state)
    else:
        state = state.repeat_interleave(S, dim=1)
    dyn_f, nu_r, item_r, dept_r = rep(dyn[:, C:]).clone(), rep(nu), rep(item), rep(dept)
    listed = rep(batch["listed"][:, C:])
    out = torch.zeros(len(series) * S, H)
    for h in range(H):                                        # 2) ancestral sampling, one step at a time
        if h > 0:
            dyn_f[:, h, 0] = out[:, h - 1] / nu_r              # feed back own sample as y_{t-1}
            dyn_f[:, h, 1] = listed[:, h - 1]
            dyn_f[:, h, 2] = 0.0                               # assume no stockout in the future
        raw, state = model(dyn_f[:, h:h + 1], item_r, dept_r, state)
        out[:, h] = model.sample(raw[:, 0], nu_r) * listed[:, h]
    return out.view(len(series), S, H).numpy()


def samples_to_frame(samples, P, origin_name, origin_t, model_name, series=None, extra=None):
    series = np.arange(P["N"]) if series is None else np.asarray(series)
    n, S, Hh = samples.shape
    df = pd.DataFrame({
        "model": model_name, "origin": origin_name,
        "sidx": np.repeat(series, Hh), "h": np.tile(np.arange(1, Hh + 1), n),
        "mean": samples.mean(1).ravel(),
    })
    for q in QUANTILES:
        df[f"q{int(q*100)}"] = np.quantile(samples, q, axis=1).ravel()
    df["t"] = origin_t + df["h"]
    for k, v in (extra or {}).items():
        df[k] = v
    return df

## Section E — Hybrid features: Kalman → DeepAR 

For every time step in a window we give the network **what the Kalman filter would have predicted with the
information available at that moment**:

* context steps ($t\le t_0$): the one-step-ahead prediction $s_{dow(t)}\hat x_{t-1|t-1}$ and its std,
* prediction steps ($t>t_0$): the $h$-step forecast $s_{dow(t)}\hat x_{t_0|t_0}$ with std $s\sqrt{P_{t_0}+hq+r}$.

Both are divided by the same scale ν as the target. Everything is causal, so no future information leaks in.

In [ ]:
def load_kalman_states():
    z = np.load(PROC / "kalman_states.npz")
    return {k: z[k] for k in z.files}


def kalman_window_features(kf, sidx, t0, tt, nu):
    s2 = sidx[:, None]
    prev = np.clip(tt - 1, 0, None)
    is_future = tt > t0[:, None]
    base_t = np.where(is_future, t0[:, None], prev)            # state used for the prediction
    steps = np.where(is_future, tt - t0[:, None], 1)            # how many steps ahead
    lvl, pv = kf["level"][s2, base_t], kf["P"][s2, base_t]
    prof = kf["prof"][s2, kf["dow"][tt]]
    q, r = kf["q"][s2], kf["r"][s2]
    mean = np.clip(prof * lvl, 0, None)
    sd = prof * np.sqrt(pv + steps * q + r)
    feats = np.stack([np.nan_to_num(mean) / nu[:, None], np.nan_to_num(sd) / nu[:, None]], -1)
    return feats.astype(np.float32)

## Section F — Evaluation metrics

* **MAE** $=\frac1n\sum|y-\hat y|$, **RMSE** $=\sqrt{\frac1n\sum(y-\hat y)^2}$
* **WAPE** $=\sum|y-\hat y| / \sum y$ — scale-free, safe with zeros (unlike MAPE)
* **Pinball loss** $L_\tau(y,\hat q)=\max(\tau(y-\hat q),(\tau-1)(y-\hat q))$, averaged over τ ∈ {0.1, 0.5, 0.9}
* **Coverage** of the 80% interval [q10, q90] (ideal = 0.80) and its mean **width**.

Point forecast = predictive **mean** for every model.

In [ ]:
def pinball(y, q, tau):
    d = y - q
    return np.maximum(tau * d, (tau - 1) * d)


def score(df):
    """df needs columns y, mean, q10, q50, q90. Rows with NaN y (not on sale) are ignored."""
    df = df[~df["y"].isna()]
    y, m = df["y"].values, df["mean"].values
    e = y - m
    pin = np.mean([pinball(y, df[f"q{int(t*100)}"].values, t).mean() for t in QUANTILES])
    return pd.Series(dict(
        MAE=np.abs(e).mean(), RMSE=np.sqrt((e ** 2).mean()), WAPE=np.abs(e).sum() / max(y.sum(), 1e-9),
        Pinball=pin, Cov80=np.mean((y >= df["q10"].values) & (y <= df["q90"].values)),
        Width80=np.mean(df["q90"].values - df["q10"].values), n=len(df)))


def attach_actuals(df, P):
    """Add the realised y, stockout flag and segment to a forecast frame."""
    df = df.copy()
    df["y"] = P["Y"][df["sidx"].values, df["t"].values]
    df["stockout"] = P["STOCK"][df["sidx"].values, df["t"].values].astype(bool)
    df["segment"] = P["meta"]["segment"].values[df["sidx"].values]
    df["dept"] = P["meta"]["dept_id"].values[df["sidx"].values]
    return df


def plot_fan(ax, P, sidx, origin_t, fc, title="", hist_days=56):
    """History + forecast mean + 80% band for one series."""
    t_hist = np.arange(origin_t - hist_days + 1, origin_t + H + 1)
    ax.plot(t_hist, P["Y"][sidx, t_hist], color="black", lw=1, label="actual")
    f = fc[(fc.sidx == sidx)].sort_values("t")
    ax.plot(f.t, f["mean"], color="C0", lw=1.5, label="mean forecast")
    ax.fill_between(f.t, f.q10, f.q90, color="C0", alpha=0.25, label="80% interval")
    st = P["STOCK"][sidx, t_hist].astype(bool)
    ax.scatter(t_hist[st], np.zeros(st.sum()), color="red", marker="x", s=15, label="stockout flag")
    ax.axvline(origin_t + 0.5, color="grey", ls="--")
    ax.set_title(title, fontsize=9)


print(f"common loaded | ROOT={ROOT} | device={DEVICE} | H={H}, C={C}, origins={ORIGINS}")